python scripts/reformat_fleury.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_fleury.tsv

In [2]:
import pandas as pd
import os

def load_table(file, separator=None):
    df = ''
    if str(file).split('.')[-1] == 'tsv':
        separator = '\t' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] == 'csv':
        separator = ',' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] in ['xls', 'xlsx']:
        df = pd.read_excel(file, index_col=None, header=0, sheet_name=0, dtype='str')
        df.fillna('', inplace=True)
    else:
        print('Wrong file format. Compatible file formats: TSV, CSV, XLS, XLSX')
        exit()
    return df

In [3]:
df_combined = pd.read_csv('../combined_fleury.tsv', sep='\t')

In [39]:
BASE_PATH = '../data/FLEURY/'
dfs = []

for file in os.listdir(BASE_PATH):
    if file.endswith(('.csv', '.tsv')):
        print(file)
        df = load_table(BASE_PATH + file, separator='\t')
        dfs.append(df)

df_full_raw = pd.concat(dfs, ignore_index=True)

20230807_GrupoFleury_Virusrespiratorios - 31-07-2023 a 06-08-2023_SE31.tsv
20230529_GrupoFleury_Virusrespiratorios - 22-05-2023 a 28-05-2023_SE21.tsv
22_06_GrupoFleury_Virusrespiratorios - 01-06-2022 a 30-06-2022.csv
23_03_GrupoFleury_Virusrespiratorios - 01-03-2023 a 31-03-2023.csv
20230522_GrupoFleury_Virusrespiratorios - 15-05-2023 a 21-05-2023_SE20.tsv
22_11_GrupoFleury_Virusrespiratorios - 01-11-2022 a 30-11-2022.csv
22_07_GrupoFleury_Virusrespiratorios - 01-07-2022 a 31-07-2022.csv
23_04_GrupoFleury_Virusrespiratorios - 01-04-2023 a 30-04-2023.csv
23_02_GrupoFleury_Virusrespiratorios - 01-02-2023 a 28-02-2023.csv
22_05_GrupoFleury_Virusrespiratorios - 01-05-2022 a 31-05-2022.csv
20230515_GrupoFleury_Virusrespiratorios - 08-05-2023 a 14-05-2023_SE19.tsv
20230731_GrupoFleury_Virusrespiratorios - 24-07-2023 a 30-07-2023_SE30.tsv
22_12_GrupoFleury_Virusrespiratorios - 01-12-2022 a 31-12-2022.csv
22_09_GrupoFleury_Virusrespiratorios - 01-09-2022 a 30-09-2022.csv
20230703_GrupoFleury_V

## Duplicatas

In [40]:
df_duplicates = (
    df_combined
    .assign( one=1 )
    .groupby(['sample_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

0 nan nan


,num_duplicates
sample_id,


In [41]:
df_duplicates = (
    df_combined
    # .query("test_kit == 'unknown'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

1 1 1


,num_duplicates
test_id,
4409835815_100,1


In [42]:
df_os_multiple_tests = (
    df_combined
    .groupby('test_id')
    .agg(kits=('test_kit',lambda x: ','.join(x.tolist())))
    .query("kits.str.contains(',')", engine='python') 
)


In [43]:
df_os_multiple_tests.head(10)

,kits
test_id,
4409835815_100,"covid_pcr,covid_pcr"


In [44]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

## Uniqueness

In [45]:
if 'unknown' in df_combined['test_kit'].unique():
    print('WARNING! unknown test kits found')

df_combined['test_kit'].unique()

array(['covid_pcr', 'covid_antigen', 'flu_antigen', 'flu_pcr', 'test_21',
       'test_4', 'vsr_antigen'], dtype=object)

In [46]:
df_combined['state'].unique()

array(['SP', 'RS', 'RJ', 'PE', 'MG', 'PR', 'AM', 'ES', 'RN', 'MS', 'GO',
       'BA', 'DF', 'MT', 'MA', 'PA', 'AC', 'SC', 'RR', 'SE', 'RO', 'TO',
       'AL', 'AP', 'CE', 'PB', 'PI'], dtype=object)

In [47]:
df_combined['location'].unique()

array(['VOTORANTIM', 'VALINHOS', 'BARUERI', ..., 'CACHOEIRA GRANDE',
       'PRESIDENTE SARNEY', 'PASTOS BONS'], dtype=object)

In [48]:
df_combined['date_testing'].max(), df_combined['date_testing'].min()

('2023-08-06', '2022-01-01')

In [49]:
df_combined['age'].unique(), df_combined['age'].min(), df_combined['age'].max()

(array([ 0,  7, 16,  1,  6, 13, 19, 20, 18, 14,  9,  8, 10,  4, 12,  5, 17,
         3,  2, 15, 11, -2]),
 -2,
 20)

In [50]:
df_combined['sex'].unique()

array(['M', 'F', '?'], dtype=object)

In [51]:
for column in df_combined.columns:
    if column.endswith('_result'):
        print(column, df_combined[column].unique())

FLUA_test_result ['NT' 'Neg' 'Pos']
FLUB_test_result ['NT' 'Neg' 'Pos']
VSR_test_result ['NT' 'Neg' 'Pos']
SC2_test_result ['Neg' 'Pos' 'NT']
META_test_result ['NT' 'Neg' 'Pos']
RINO_test_result ['NT' 'Neg' 'Pos']
PARA_test_result ['NT' 'Neg' 'Pos']
ADENO_test_result ['NT' 'Neg' 'Pos']
BOCA_test_result ['NT']
COVS_test_result ['NT' 'Neg' 'Pos']
ENTERO_test_result ['NT']
BAC_test_result ['NT' 'Neg' 'Pos']


In [52]:
for col in df_combined.columns:
    if col.endswith('_result'):
        print(col)

FLUA_test_result
FLUB_test_result
VSR_test_result
SC2_test_result
META_test_result
RINO_test_result
PARA_test_result
ADENO_test_result
BOCA_test_result
COVS_test_result
ENTERO_test_result
BAC_test_result


## Inconsistencies

In [53]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

Verificando se um mesmo test_id + test_kit apresentam mais de um resultado para um mesmo patógeno

In [54]:
agg_test_rules = {
    col+'_kit': (col, lambda x: ','.join(x.tolist()))
    for col in df_combined.columns
    if col.endswith('_result')
}

df_os_non_contraditory_test_results = (
    df_combined
    .groupby(['test_id', 'test_kit'])
    .agg(**agg_test_rules)
)

In [55]:
for col in df_os_non_contraditory_test_results.columns:
    if col.endswith('_kit'):
        print(col, df_os_non_contraditory_test_results[col].unique())
        # Devem ser apenas ['NT' 'Neg' 'Pos']

FLUA_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
FLUB_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
VSR_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
SC2_test_result_kit ['Neg' 'Pos' 'NT' 'Neg,Neg']
META_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']


RINO_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
PARA_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
ADENO_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
BOCA_test_result_kit ['NT' 'NT,NT']
COVS_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']
ENTERO_test_result_kit ['NT' 'NT,NT']
BAC_test_result_kit ['NT' 'Neg' 'Pos' 'NT,NT']


NOTA: Esse teste vai falhar em um único exemplo do conjunto de dados.

O comportamento esperado é que um mesmo test_id seja utilizado apenas em um único test_kit ou em vários test_kit DIFERENTES.

No caso do exemplo 4409835815_100, o mesmo test_id foi utilizado duas vezes para um mesmo test_kit (covid_pcr)

In [70]:
df_os_non_contraditory_test_results.query("SC2_test_result_kit == 'Neg,Neg'")

,,FLUA_test_result_kit,FLUB_test_result_kit,VSR_test_result_kit,SC2_test_result_kit,META_test_result_kit,RINO_test_result_kit,PARA_test_result_kit,ADENO_test_result_kit,BOCA_test_result_kit,COVS_test_result_kit,ENTERO_test_result_kit,BAC_test_result_kit
test_id,test_kit,,,,,,,,,,,,
4409835815_100,covid_pcr,"NT,NT","NT,NT","NT,NT","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"


In [71]:
df_full_raw.query("`CODIGO REQUISICAO` == '4409835815_100'")

,CODIGO REQUISICAO,DATA COLETA,PACIENTE,SEXO,IDADE,MUNICIPIO,ESTADO,EXAME,PATOGENO,RESULTADO
524415,4409835815_100,08/05/2023,25A2DED6E5723557D3C00CEBC6E4E42A,M,7M9D,SAO PAULO,SP,2019NCOV,"Covid 19, Detecção por PCR",NÃO DETECTADO (NEGATIVO)
787051,4409835815_100,06/05/2023,25A2DED6E5723557D3C00CEBC6E4E42A,M,7M9D,SAO PAULO,SP,2019NCOV,"Covid 19, Detecção por PCR",NÃO DETECTADO (NEGATIVO)


In [72]:
df_combined.query("test_id == '4409835815_100'")

,lab_id,test_id,test_kit,patient_id,sample_id,state,location,date_testing,epiweek,age,...,Ct_RDRP,geneS_detection,META_test_result,RINO_test_result,PARA_test_result,ADENO_test_result,BOCA_test_result,COVS_test_result,ENTERO_test_result,BAC_test_result
614194,FLEURY,4409835815_100,covid_pcr,25A2DED6E5723557D3C00CEBC6E4E42A,46aab5430ae7ebcc,SP,SAO PAULO,2023-05-06,2023-05-06,0,...,NaN,NaN,NT,NT,NT,NT,NT,NT,NT,NT
614195,FLEURY,4409835815_100,covid_pcr,25A2DED6E5723557D3C00CEBC6E4E42A,4b9675bba936599b,SP,SAO PAULO,2023-05-08,2023-05-13,0,...,NaN,NaN,NT,NT,NT,NT,NT,NT,NT,NT


### Ids

Verificando se todos os Ids originais etão presentes no arquivo final

In [60]:
set_test_ids_final_df = set(df_combined['test_id'])
set_test_ids_raw_data = set(df_full_raw['CODIGO REQUISICAO'])

In [61]:
# Testar se todos os test_ids da planilha original estão na planilha final
# deve retornar um set vazio
set_test_ids_raw_data.difference(set_test_ids_final_df)

{'1200085178_100',
 '1410231927_100',
 '1421499943_100',
 '1421512238_100',
 '1421512252_100',
 '1421524508_100',
 '1421542791_100',
 '1421567484_600',
 '1421573025_900',
 '1421574120_100',
 '1421580321_100',
 '1421580345_100',
 '1421580359_600',
 '1421580372_100',
 '1421580414_700',
 '1421590360_100',
 '1421608991_700',
 '1421618661_100',
 '1421623648_100',
 '1421627267_100',
 '1421641149_200',
 '1421661054_100',
 '1421747497_400',
 '1450064023_100',
 '1450064862_100',
 '1600683873_100',
 '1600708233_100',
 '1600709104_100',
 '1600712587_100',
 '1650441468_100',
 '1650447361_100',
 '1650451652_100',
 '1660378198_100',
 '1720397744_100',
 '1860281386_100',
 '1870163611_100',
 '2230148082_100',
 '2240159856_100',
 '2240169073_100',
 '2240170521_100',
 '2240193197_100',
 '2240197205_100',
 '2310036024_100',
 '2320523931_100',
 '2320536507_100',
 '2320542723_100',
 '2320550391_100',
 '2320551123_100',
 '2320551273_100',
 '2320562210_100',
 '2320582241_100',
 '2350037816_100',
 '2350041791

In [62]:
# Verificar se há test_ids na planilha final que não estão na planilha original
# deve retornar um set vazio
set_test_ids_final_df.difference(set_test_ids_raw_data)

set()

Verificando se todos os patógenos de um test_kit estão sendo testados para todos os test_id

In [91]:
PATHOGENS_TESTS = {
    'panel_21':{
        # PAINCOVI
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'COVS_test_result',
        'BAC_test_result',
    },

    'panel_24':{
        # RESPIRA
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'BOCA_test_result',
        'COVS_test_result',
        'ENTERO_test_result',
        'BAC_test_result',
    },

    'flu_antigen':{
        'FLUA_test_result',
        'FLUB_test_result',
    },

    'covid_antigen':{
        'SC2_test_result',
    },

    'covid_pcr':{
        'SC2_test_result',
    },
}

In [92]:
# Mapeando Pos e Neg para Tes
df_combined_test_pathogen_non_nt_on_kit = (
    df_combined
    .replace(('Pos', 'Neg'),  'Tes')
)

In [93]:
for pathogen, test_columns in PATHOGENS_TESTS.items():
    print(pathogen)
    print(
        df_combined_test_pathogen_non_nt_on_kit
        .query("test_kit == @pathogen")
        [list(test_columns)]
        .drop_duplicates().T
        # Deve resultar em apenas uma linha completa por 'Tes'
    )

    print('\n\n')

panel_21
Empty DataFrame
Columns: []
Index: [BAC_test_result, ADENO_test_result, SC2_test_result, FLUA_test_result, META_test_result, PARA_test_result, FLUB_test_result, COVS_test_result, RINO_test_result, VSR_test_result]



panel_24
Empty DataFrame
Columns: []
Index: [BAC_test_result, BOCA_test_result, ADENO_test_result, ENTERO_test_result, FLUA_test_result, META_test_result, PARA_test_result, FLUB_test_result, COVS_test_result, RINO_test_result, VSR_test_result]



flu_antigen
                   70
FLUB_test_result  Tes
FLUA_test_result  Tes



covid_antigen
                   2
SC2_test_result  Tes



covid_pcr
                0      733073
SC2_test_result    Tes     NT





Verificando resultados de alguns testes

In [103]:
import numpy as np

In [104]:
np.random.seed(214)

ids = np.random.choice(df_combined['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [105]:
current_id=50

3560739792_100

In [134]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_full_raw.query("`CODIGO REQUISICAO` == @id") [['CODIGO REQUISICAO', 'EXAME', 'RESULTADO']].T )

                             670264
test_id             4452122163_1100
test_kit                  covid_pcr
FLUA_test_result                 NT
FLUB_test_result                 NT
VSR_test_result                  NT
SC2_test_result                 Pos
META_test_result                 NT
RINO_test_result                 NT
PARA_test_result                 NT
ADENO_test_result                NT
BOCA_test_result                 NT
COVS_test_result                 NT
ENTERO_test_result               NT
BAC_test_result                  NT
                                 826036
CODIGO REQUISICAO       4452122163_1100
EXAME                          2019NCOV
RESULTADO          DETECTADO (POSITIVO)


Verificando resultados de alguns testes - Fltrando por test_kit

In [135]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit not in ('covid_pcr', 'covid_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [143]:
current_id=0

In [196]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_full_raw.query("`CODIGO REQUISICAO` == @id") [['CODIGO REQUISICAO', 'EXAME', 'PATOGENO','RESULTADO']].T )

                           1016398
test_id             8705357837_100
test_kit                   test_21
FLUA_test_result               Neg
FLUB_test_result               Neg
VSR_test_result                Pos
SC2_test_result                Neg
META_test_result               Neg
RINO_test_result               Neg
PARA_test_result               Neg
ADENO_test_result              Neg
BOCA_test_result                NT
COVS_test_result               Pos
ENTERO_test_result              NT
BAC_test_result                Neg
                                   562381                        562382  \
CODIGO REQUISICAO          8705357837_100                8705357837_100   
EXAME                            VIRUSMOL                      VIRUSMOL   
PATOGENO           Virusmol, Adenovï¿½rus  Virusmol, Coronavï¿½rus 229E   
RESULTADO                        NEGATIVO                      NEGATIVO   

                                         562383                        562384  \
CODIGO REQUISICAO 

In [197]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit in ('flu_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [247]:
current_id=0

In [248]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_full_raw.query("`CODIGO REQUISICAO` == @id") [['CODIGO REQUISICAO', 'EXAME', 'PATOGENO','RESULTADO']].T )

                            609116
test_id             4403728288_300
test_kit               flu_antigen
FLUA_test_result               Neg
FLUB_test_result               Neg
VSR_test_result                 NT
SC2_test_result                 NT
META_test_result                NT
RINO_test_result                NT
PARA_test_result                NT
ADENO_test_result               NT
BOCA_test_result                NT
COVS_test_result                NT
ENTERO_test_result              NT
BAC_test_result                 NT
                                           803954
CODIGO REQUISICAO                  4403728288_300
EXAME                                     AGINFLU
PATOGENO           Influenza A e B - teste rápido
RESULTADO                                NEGATIVO
